In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

In [0]:
# Load the test data and GBT scores
test = spark.table("workspace.mortgage_risk.model_test")

score_columns = [
    "loan_id",
    "prediction_month",
    "label_12m",
    "delinquency_months",
    "risk_score"
]

gbt_scores = spark.table(
    "workspace.mortgage_risk.test_scores_gbt_v1"
).select(*score_columns)

model_folder = "/Volumes/workspace/mortgage_risk/model_artifacts"
lr_score_table = "workspace.mortgage_risk.test_scores_lr_v1"

In [0]:
# Generate logistic regression predictions
if spark.catalog.tableExists(lr_score_table):
    lr_scores = spark.table(lr_score_table).select(*score_columns)
    print("Loaded the saved logistic regression scores.")

else:
    lr_model = PipelineModel.load(
        model_folder + "/logistic_regression_v1"
    )

    lr_predictions = lr_model.transform(test)

    lr_scores = lr_predictions.withColumn(
        "risk_score",
        vector_to_array("probability")[1]
    ).select(*score_columns)

    lr_scores.write.format("delta").mode("error").saveAsTable(
        lr_score_table
    )

    lr_scores = spark.table(lr_score_table).select(*score_columns)
    print("Logistic regression test scores saved.")

Loaded the saved logistic regression scores.


In [0]:
# Check that both models use the same observations
cohort_columns = [
    "loan_id",
    "prediction_month",
    "label_12m",
    "delinquency_months"
]

gbt_cohort = gbt_scores.select(*cohort_columns)
lr_cohort = lr_scores.select(*cohort_columns)

assert gbt_cohort.exceptAll(lr_cohort).limit(1).count() == 0, \
    "Some GBT observations are missing or different in LR."

assert lr_cohort.exceptAll(gbt_cohort).limit(1).count() == 0, \
    "Some LR observations are missing or different in GBT."

print("Both models use the same test observations and labels.")

Both models use the same test observations and labels.


In [0]:
# Combine the models and prepare both populations
gbt_scores = gbt_scores.withColumn(
    "model", F.lit("GBT: 50 trees")
)

lr_scores = lr_scores.withColumn(
    "model", F.lit("Logistic regression")
)

model_scores = gbt_scores.unionByName(lr_scores)

alert_scores = model_scores.withColumn(
    "population", F.lit("All eligible loans")
)

healthy_scores = model_scores.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population", F.lit("Current status 00")
)

alert_scores = alert_scores.unionByName(healthy_scores)

In [0]:
# Rank each model separately each month
month_window = Window.partitionBy(
    "model",
    "population",
    "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

ranked = alert_scores.withColumn(
    "monthly_rows",
    F.count("*").over(month_window)
)

ranked = ranked.withColumn(
    "risk_rank",
    F.row_number().over(risk_window)
)

In [0]:
# Flag the highest-risk 5%
ranked = ranked.withColumn(
    "alert",
    F.when(
        F.col("risk_rank") <= F.ceil(F.col("monthly_rows") * 0.05),
        1
    ).otherwise(0)
)

ranked = ranked.withColumn(
    "event_caught",
    F.when(
        F.col("alert") == 1,
        F.col("label_12m")
    ).otherwise(0)
)

In [0]:
# Count the monthly results
monthly_counts = ranked.groupBy(
    "model",
    "population",
    "prediction_month"
).agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.sum("alert").alias("alerts"),
    F.sum("event_caught").alias("events_caught")
)

In [0]:
#. Save the monthly counts
monthly_count_table = ("workspace.mortgage_risk.test_monthly_alert_counts_gbt_lr_v1")

if not spark.catalog.tableExists(monthly_count_table):
    monthly_counts.write.format("delta").mode("error").saveAsTable(
        monthly_count_table)

monthly_counts = spark.table(monthly_count_table)

In [0]:
# Show the overall comparison
comparison = monthly_counts.groupBy("model", "population").agg(
    F.sum("rows").alias("rows"),
    F.sum("events").alias("events"),
    F.sum("alerts").alias("alerts"),
    F.sum("events_caught").alias("events_caught")
)

comparison = comparison.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")
    )
)

comparison = comparison.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")
    )
)

comparison = comparison.withColumn(
    "lift",
    F.when(
        F.col("events") > 0,
        F.col("precision") / (F.col("events") / F.col("rows"))
    )
)

display(comparison.orderBy("population", "model"))

model,population,rows,events,alerts,events_caught,precision,recall,lift
GBT: 50 trees,All eligible loans,3306965,25184,165358,13958,0.08441079355096215,0.5542407878017789,11.084162162295804
Logistic regression,All eligible loans,3306965,25184,165358,13033,0.0788168700637405,0.5175111181702668,10.349612083479098
GBT: 50 trees,Current status 00,3275008,16048,163757,5358,0.03271921200315101,0.33387337986041876,6.677198471087712
Logistic regression,Current status 00,3275008,16048,163757,4362,0.026637029256764595,0.2718095712861416,5.435972327526054


In [0]:
                    ####check whether that advantage holds across the test months####

In [0]:
monthly_counts = spark.table("workspace.mortgage_risk.test_monthly_alert_counts_gbt_lr_v1")

In [0]:
monthly_comparison = monthly_counts.withColumn(
    "event_rate",
    F.col("events") / F.col("rows"))

monthly_comparison = monthly_comparison.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")))

monthly_comparison = monthly_comparison.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")))

monthly_comparison = monthly_comparison.withColumn(
    "lift",
    F.when(
        F.col("events") > 0,
        F.col("precision") / F.col("event_rate")))

display(
    monthly_comparison.orderBy(
        "population",
        "prediction_month",
        "model"
    )
)

model,population,prediction_month,rows,events,alerts,events_caught,event_rate,precision,recall,lift
GBT: 50 trees,All eligible loans,2024-01-01,201325,1451,10067,780,0.007207251955792872,0.07748087811661865,0.5375603032391454,10.750405090853377
Logistic regression,All eligible loans,2024-01-01,201325,1451,10067,701,0.007207251955792872,0.06963345584583291,0.48311509303928324,9.661582011138739
GBT: 50 trees,All eligible loans,2024-02-01,204642,1507,10233,795,0.007364079709932468,0.07768982703019642,0.5275381552753815,10.549835158004946
Logistic regression,All eligible loans,2024-02-01,204642,1507,10233,747,0.007364079709932468,0.07299912049252419,0.49568679495686796,9.912863978653705
GBT: 50 trees,All eligible loans,2024-03-01,208027,1552,10402,823,0.007460570022160585,0.07911940011536243,0.5302835051546392,10.605007376158829
Logistic regression,All eligible loans,2024-03-01,208027,1552,10402,758,0.007460570022160585,0.07287060180734474,0.4884020618556701,9.767430851917851
GBT: 50 trees,All eligible loans,2024-04-01,210560,1602,10528,863,0.007608282674772036,0.08197188449848024,0.5387016229712859,10.774032459425717
Logistic regression,All eligible loans,2024-04-01,210560,1602,10528,808,0.007608282674772036,0.07674772036474165,0.5043695380774033,10.087390761548066
GBT: 50 trees,All eligible loans,2024-05-01,213883,1657,10695,901,0.007747226287269208,0.08424497428705002,0.5437537718768859,10.874211125791865
Logistic regression,All eligible loans,2024-05-01,213883,1657,10695,827,0.007747226287269208,0.07732585320243104,0.49909474954737476,9.981101665959903


In [0]:
results_to_save = [(comparison,"workspace.mortgage_risk.test_alert_comparison_gbt_lr_v1"),
    (monthly_comparison,"workspace.mortgage_risk.test_monthly_alert_comparison_gbt_lr_v1")]

for results, table_name in results_to_save:
    if spark.catalog.tableExists(table_name):
        print("Already saved:", table_name)
    else:
        results.write.format("delta").mode("error").saveAsTable(
            table_name
        )
        print("Saved:", table_name)

Saved: workspace.mortgage_risk.test_alert_comparison_gbt_lr_v1
Saved: workspace.mortgage_risk.test_monthly_alert_comparison_gbt_lr_v1
